
# CAP4453 Assignment 2 — **Two‑Layer CNN (Minimal Template)**

This notebook is a **bare‑bones scaffold** for your Assignment 2 experiments.  
It includes **only** a two‑layer CNN example and **clearly marked** `# STUDENT TODO:`:
- **Data augmentation**
- **Image normalisation** (in the dataloader)
- **Batch Normalisation** (inside the model)
- **Optimizer & LR scheduler**
- **Training / validation / test loops**
- **Plots & analysis (e.g., misclassified examples)**

> **Important:** You must **design any deeper CNNs yourself** (e.g., 3‑layer or more). Do **not** expect ready‑made models here. You could design it in this notebook or in any other notebook/python files.



## Next steps
- **Architectural variations:** Modify the number of filters (`hidden_channels1`, `hidden_channels2`), change kernel sizes, or try different pooling strategies.
- **Make CNN deeper:** Add one or two extra convolutional layers. Does a deeper network improve accuracy?
- **Data augmentation:** Add transforms such as random cropping, horizontal flipping, colour jitter, and small rotations.
- **Data Normalisation:** Use `transforms.Normalize(mean, std)` in the **dataloader** after `ToTensor()`.
- **Batch Normalisation:** Insert `nn.BatchNorm2d(num_channels)` **after** `nn.Conv2d` and **before** `nn.ReLU()` in each block.
- **Comparison with Assignment 1:** Train your best 2‑layer and 3-layer CNNs and report test/val accuracy. Compare to Assignment 1’s linear and two‑layer fully connected baselines.
- **Analysis:** Identify misclassified examples and discuss patterns.


## 1. Setup (imports, device, seeds)

In [1]:
# Minimal imports
import os, random, time
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

# Reproducibility (optional)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Device (CPU/GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


Using device: cuda



## 2. Datasets & Transforms (augmentation + image normalisation)

- Switch dataset with `dataset_name`.
- **Add augmentation** inside the marked list(s).
- **Add image normalisation** via `transforms.Normalize(mean, std)` **after** `transforms.ToTensor()`.
- Keep augmentation **ON** for train only; for val/test typically only `ToTensor()` and **normalisation**.


In [ ]:
# STUDENT TODO: choose dataset and flags
dataset_name = 'cifar10'  # 'cifar10' or 'mnist'  (STUDENT TODO)
AUGMENT      = True      # turn ON/OFF augmentation (STUDENT TODO)
USE_NORMALIZE= True      # turn ON/OFF image normalisation (STUDENT TODO)

# STUDENT TODO: adjust batch size and val split
batch_size = 128
val_ratio  = 0.1

# --- Build transforms (MINIMAL default: ToTensor only) ---
if dataset_name.lower() == 'cifar10':
    # TRAIN transforms (add augmentation in the list below)
    train_tfms_list = []
    if AUGMENT:
        # STUDENT TODO (AUGMENT): add items such as RandomCrop/HorizontalFlip/ColorJitter/Rotation, etc.
        train_tfms_list += [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(p=0.5), transforms.ColorJitter(brightness=0.25, contrast=0.25), transforms.RandomRotation(10)]
    train_tfms_list += [transforms.ToTensor()]  # required
    
    if USE_NORMALIZE:
        # STUDENT TODO (IMAGE NORMALISATION): typical CIFAR-10 stats
        train_tfms_list += [transforms.Normalize(mean=(0.4914,0.4822,0.4465), std=(0.2470,0.2435,0.2616))]
    
    test_tfms_list = [transforms.ToTensor()]
    if USE_NORMALIZE:
        test_tfms_list += [transforms.Normalize(mean=(0.4914,0.4822,0.4465), std=(0.2470,0.2435,0.2616))]
    
    train_tfms = transforms.Compose(train_tfms_list)
    test_tfms  = transforms.Compose(test_tfms_list)

    train_dataset = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_tfms)
    test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_tfms)
    num_classes = 10
    in_channels = 3

elif dataset_name.lower() == 'mnist':
    train_tfms_list = []
    if AUGMENT:
        # STUDENT TODO (AUGMENT for MNIST): small rotations/affine
        # e.g., transforms.RandomRotation(10)
        pass
    train_tfms_list += [transforms.ToTensor()]
    
    if USE_NORMALIZE:
        # STUDENT TODO (IMAGE NORMALISATION): typical MNIST stats
        # train_tfms_list += [transforms.Normalize(mean=(0.1307,), std=(0.3081,))]
        pass
    
    test_tfms_list = [transforms.ToTensor()]
    if USE_NORMALIZE:
        # STUDENT TODO: same normalisation as train
        # test_tfms_list += [transforms.Normalize(mean=(0.1307,), std=(0.3081,))]
        pass
    
    train_tfms = transforms.Compose(train_tfms_list)
    test_tfms  = transforms.Compose(test_tfms_list)

    train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=train_tfms)
    test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=test_tfms)
    num_classes = 10
    in_channels = 1
else:
    raise ValueError('Unknown dataset: choose cifar10 or mnist')

# --- Train/Val split ---
train_size = int((1.0 - val_ratio) * len(train_dataset))
val_size   = len(train_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_dataset)}')


100%|██████████| 170M/170M [00:04<00:00, 38.6MB/s] 


Train: 45000 | Val: 5000 | Test: 10000



## 3. Two‑Layer CNN (only) — add BatchNorm where marked

> You must create **any deeper CNNs** in a **separate class** (your own file or below), using this as a starting point.


In [ ]:
class TwoLayerCNN(nn.Module):
    """A minimal two‑layer CNN.
    Block1: Conv -> (BatchNorm?) -> ReLU -> Pool
    Block2: Conv -> (BatchNorm?) -> ReLU -> Pool
    FC:     Linear to num_classes (lazy‑init after first forward)
    """
    def __init__(self, in_channels, num_classes, hidden_channels1=16, hidden_channels2=32, use_bn=False):
        super().__init__()
        self.use_bn = use_bn
        self.in_channels = in_channels
        self.num_classes = num_classes
        # --- Block 1 ---
        self.conv1 = nn.Conv2d(in_channels, hidden_channels1, kernel_size=3, padding=1, bias=not use_bn)
        # STUDENT TODO (BATCH NORM): if use_bn, insert nn.BatchNorm2d(hidden_channels1) before ReLU
        self.bn1 = nn.Identity()
        if use_bn:
            self.bn1 = nn.BatchNorm2d(hidden_channels1)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2)

        # --- Block 2 ---
        self.conv2 = nn.Conv2d(hidden_channels1, hidden_channels2, kernel_size=3, padding=1, bias=not use_bn)
        # STUDENT TODO (BATCH NORM): if use_bn, insert nn.BatchNorm2d(hidden_channels2) before ReLU
        self.bn2 = nn.Identity()
        if use_bn:
            self.bn2 = nn.BatchNorm2d(hidden_channels2)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2)

        # Classifier head will be created lazily (first forward) so you don't need to compute shapes by hand
        self.classifier = None
        self.num_classes = num_classes

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        if self.classifier is None:
            self.classifier = nn.Linear(x.size(1), self.num_classes).to(x.device)
        return self.classifier(x)

# STUDENT TODO: create your OWN deeper CNN class (e.g., ThreeLayerCNN) in another cell/file.
# Do NOT expect a prewritten deeper network here.


In [4]:
class ThreeLayerCNN(nn.Module):
    def __init__(self, in_channels, num_classes, hidden_channels1=16, hidden_channels2=32, hidden_channels3=64, use_bn=False):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.hidden_channels1 = hidden_channels1
        self.conv1 = nn.Conv2d(in_channels, hidden_channels1, kernel_size=3, padding=1, bias=not use_bn)
        self.relu = nn.ReLU(inplace=True)
        self.hidden_channels2 = hidden_channels2
        self.conv2 = nn.Conv2d(hidden_channels1, hidden_channels2, kernel_size=3, padding=1, bias=not use_bn)
        self.hidden_channels3 = hidden_channels3
        self.conv3 = nn.Conv2d(hidden_channels2, hidden_channels3, kernel_size=3, padding=1, bias = not use_bn)
        self.bn1 = nn.Identity()
        self.bn2 = nn.Identity()
        self.bn3 = nn.Identity()
        if use_bn:
            self.bn1 = nn.BatchNorm2d(hidden_channels1)
            self.bn2 = nn.BatchNorm2d(hidden_channels2)
            self.bn3 = nn.BatchNorm2d(hidden_channels3)
        self.pool = nn.MaxPool2d(2)
        self.classifier = None

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        if self.classifier is None:
            self.classifier = nn.Linear(x.size(1), self.num_classes).to(x.device)
        return self.classifier(x)



## 4. Optimizer & (optional) LR scheduler

In [ ]:
learning_rate = 1e-3      
weight_decay  = 1e-4      
momentum      = 0.9      

model1 = TwoLayerCNN(3, 10, 32, 64, use_bn=True).to(device)  # STUDENT TODO: toggle use_bn, channels
model2 = ThreeLayerCNN(3, 10,32, 64, 128, use_bn=True).to(device) # Updated to more hidden channels
# Example choices (uncomment ONE):
# optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum, weight_decay=weight_decay)  # STUDENT TODO
optimizer1 = torch.optim.Adam(model1.parameters(), lr=learning_rate, weight_decay=weight_decay)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=learning_rate, weight_decay=weight_decay)                    # STUDENT TODO
                    # STUDENT TODO
# optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay, momentum=momentum)  # STUDENT TODO

# STUDENT TODO (optional): uncomment LR scheduler for better accuracy
scheduler1 = torch.optim.lr_scheduler.StepLR(optimizer1, step_size=20, gamma=0.1)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=20, gamma=0.1)


criterion = nn.CrossEntropyLoss()
print(model1)
print(model2)


TwoLayerCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU(inplace=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU(inplace=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
ThreeLayerCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

## 5. Training / Validation — skeleton to complete

In [6]:
# STUDENT TODO: implement standard training & evaluation
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        predicted_items = logits.argmax(dim = 1)
        correct += (predicted_items == y).sum().item()
        total_samples += len(y)
        loss = criterion(logits, y)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_loss= total_loss / len(loader)
    avg_acc = (correct / total_samples) * 100
    return avg_loss, avg_acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    # Pseudocode / hints:
    total_loss = 0
    correct = 0
    total_entries = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        predicted = logits.argmax(dim=1)
        correct += (predicted == y).sum().item()
        total_loss += loss.item()
        total_entries += len(y)
    avg_loss = total_loss / len(loader)
    avg_acc = (correct / total_entries) * 100
    return avg_loss, avg_acc


## 6. Run training (log metrics, make plots)

In [8]:
num_epochs = 30  # STUDENT TODO

train1_hist, val1_hist = [], [] # Two Layer CNN loss / accuracies
train2_hist, val2_hist = [], []
# STUDENT TODO: uncomment after implementing train/eval
for epoch in range(1, num_epochs + 1):
      tr1_loss, tr1_acc = train_one_epoch(model1, train_loader, optimizer1, criterion, device)
      tr2_loss, tr2_acc = train_one_epoch(model2, train_loader, optimizer2, criterion, device)
      va1_loss, va1_acc = evaluate(model1, val_loader, criterion, device)
      va2_loss, va2_acc = evaluate(model2, val_loader, criterion, device)
      train1_hist.append((tr1_loss, tr1_acc))
      val1_hist.append((va1_loss, va1_acc))
      train2_hist.append((tr2_loss, tr2_acc))
      val2_hist.append((va2_loss, va2_acc))
      scheduler1.step()
      scheduler2.step()
      print(f'Epoch {epoch:02d}/{num_epochs} | Train1 {tr1_loss:.4f}/{tr1_acc:.2f}% | Val1 {va1_loss:.4f}/{va1_acc:.2f}%')
      print(f'Epoch {epoch:02d}/{num_epochs} | Train2 {tr2_loss:.4f}/{tr2_acc:.2f}% | Val2 {va2_loss:.4f}/{va2_acc:.2f}%')
epochs = range(1, num_epochs + 1)
# STUDENT TODO: Create plots (loss/acc curves) using matplotlib
# Hint: use lists in train_hist/val_hist. Example keys: [x[0] for x in train_hist] for loss, [x[1] ...] for acc.
train1_loss = [x[0] for x in train1_hist] # This is the loss values across each epoch for training set
val1_loss = [x[0] for x in val1_hist] # This is the loss values across each epoch for validation set

# Plot for model1 train vs val loss
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(epochs, train1_loss, label='Train Loss', color='blue')
axes[0, 0].plot(epochs, val1_loss,   label='Val Loss',   color='orange', linestyle='--')
axes[0, 0].set_title('TwoLayerCNN — Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)


train1_acc = [x[1] for x in train1_hist] # This is the accuracies for each epoch for training set
val1_acc = [x[1] for x in val1_hist] # This is the accuracies for each epoch for validation set
# Plot for model1 train vs val accuracy
axes[0, 1].plot(epochs, train1_acc, label='Train Acc', color='blue')
axes[0, 1].plot(epochs, val1_acc,   label='Val Acc',   color='orange', linestyle='--')
axes[0, 1].set_title('TwoLayerCNN — Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend()
axes[0, 1].grid(True)

train2_loss = [x[0] for x in train2_hist]
val2_loss = [x[0] for x in val2_hist]
axes[1, 0].plot(epochs, train2_loss, label='Train Loss', color='green')
axes[1, 0].plot(epochs, val2_loss,   label='Val Loss',   color='red', linestyle='--')
axes[1, 0].set_title('ThreeLayerCNN — Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

train2_acc = [x[1] for x in train2_hist]
val2_acc = [x[1] for x in val2_hist]
axes[1, 1].plot(epochs, train2_acc, label='Train Acc', color='green')
axes[1, 1].plot(epochs, val2_acc,   label='Val Acc',   color='red', linestyle='--')
axes[1, 1].set_title('ThreeLayerCNN — Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.pdf', format='pdf', bbox_inches='tight')
plt.show()
print('Saved to training_curves.pdf')


Epoch 01/30 | Train1 1.5994/44.49% | Val1 1.5837/44.82%
Epoch 01/30 | Train2 0.9148/68.79% | Val2 0.9231/68.62%
Epoch 02/30 | Train1 1.5920/44.54% | Val1 1.5987/44.92%
Epoch 02/30 | Train2 0.9128/68.89% | Val2 0.9308/67.62%
Epoch 03/30 | Train1 1.5947/44.36% | Val1 1.5877/44.96%
Epoch 03/30 | Train2 0.9124/69.02% | Val2 0.9222/67.56%
Epoch 04/30 | Train1 1.5942/44.33% | Val1 1.5831/46.44%
Epoch 04/30 | Train2 0.9118/68.93% | Val2 0.9195/68.24%


KeyboardInterrupt: 

## 7. Analysis — misclassified examples (optional)

In [ ]:
# STUDENT TODO: display a small grid of misclassified images
# Hints:
# - collect logits, compare argmax with true labels, store a few errors
# - if you used Normalize, consider an inverse normalisation for display
# - use matplotlib to show images with predicted/true labels
